# Loan Application Evaluator — Parallel LangGraph Workflow

## Architecture
```
                 ┌──→ credit_evaluation ──→┐
                 │                          │
START ───────────┼──→ income_verification ──┼──→ aggregator ──→ summarizer ──→ END
                 │                          │
                 └──→ employment_evaluation─┘
```

**Key Design Decisions:**
- Three specialist LLM agents run in PARALLEL (credit, income, employment)
- Aggregator uses PURE LOGIC — no LLM — to combine scores and make preliminary decision
- Summarizer calls LLM ONCE to generate final human-readable verdict
- This ensures aggregator runs only ONCE after all three parallel nodes complete
- Total LLM calls: 4 (3 specialists + 1 summarizer)

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import json

load_dotenv()
model = ChatOpenAI()

## State Schema

In LangGraph, ALL fields that any node will ever read or write must be declared upfront in the State class.
Think of it as the shared memory of the entire graph.

In [2]:
class LoanApplicationState(TypedDict):
    # ── Input fields (provided by caller) ──────────────────────────
    applicant_name: str
    credit_score: int          # e.g. 420, 720, 800
    loan_amount: float         # total loan requested
    monthly_income: float      # applicant's monthly take-home
    employment_years: int      # years at current employer
    employment_type: str       # full-time / part-time / contract / self-employed

    # ── Parallel node outputs (written by specialist agents) ────────
    credit_score_result: str   # raw JSON string from credit agent
    income_score_result: str   # raw JSON string from income agent
    employment_result: str     # raw JSON string from employment agent

    # ── Aggregator outputs (written by aggregator, no LLM) ─────────
    credit_parsed: dict        # parsed credit agent result
    income_parsed: dict        # parsed income agent result
    employment_parsed: dict    # parsed employment agent result
    all_scores: list           # [credit_score, income_score, employment_score]
    preliminary_decision: str  # APPROVE / REJECT / MANUAL_REVIEW

    # ── Summarizer outputs (written by summarizer LLM) ─────────────
    verdict: str               # final APPROVE / REJECT / MANUAL_REVIEW
    summary: str               # human-readable explanation for applicant
    key_concern: str           # main risk factor identified

## Node Definitions

### Node 1 — Credit Evaluation (LLM)
Evaluates credit score against loan amount. Acts as a credit risk analyst.

In [3]:
def credit_evaluation(state: LoanApplicationState) -> dict:
    """
    Specialist Agent 1: Credit Risk Analyst
    Evaluates applicant credit score against loan amount.
    Returns score 0-10 + reasoning + risk level.
    """
    prompt = f"""
You are a credit risk analyst at a bank.

Applicant credit score : {state['credit_score']}
Loan amount requested  : {state['loan_amount']}

Industry standard:
  750+      → Excellent
  700–750   → Good
  650–700   → Fair
  Below 650 → Poor

Evaluate the credit worthiness for this loan amount.
Give a score between 0 and 10.
Give a one-line reasoning.

Respond ONLY in raw JSON. No markdown, no extra text:
{{"score": <number>, "reasoning": "<text>", "risk_level": "<LOW/MEDIUM/HIGH>"}}
"""
    response = model.invoke(prompt)
    print(f"[Credit Agent] → {response.content}")
    return {"credit_score_result": response.content}

### Node 2 — Income Verification (LLM)
Calculates debt-to-income ratio and evaluates if income supports the loan.

In [4]:
def income_verification(state: LoanApplicationState) -> dict:
    """
    Specialist Agent 2: Loan Underwriter
    Evaluates monthly income against estimated EMI.
    Returns score 0-10 + reasoning + DTI ratio.
    """
    estimated_emi = state["loan_amount"] / 60  # assuming 5-year loan tenure

    prompt = f"""
You are a loan underwriter at a bank.

Applicant monthly income : {state['monthly_income']}
Estimated monthly EMI    : {estimated_emi:.2f}  (loan / 60 months)

Calculate the debt-to-income (DTI) ratio = (EMI / monthly income) * 100.
Industry safe threshold: DTI below 40% is acceptable.

Evaluate whether the income adequately supports this loan.
Give a score between 0 and 10.
Give a one-line reasoning.

Respond ONLY in raw JSON. No markdown, no extra text:
{{"score": <number>, "reasoning": "<text>", "dti_ratio": "<percentage e.g. 35.5%>"}}
"""
    response = model.invoke(prompt)
    print(f"[Income Agent] → {response.content}")
    return {"income_score_result": response.content}

### Node 3 — Employment Evaluation (LLM)
Evaluates job stability and income reliability based on employment profile.

In [5]:
def employment_evaluation(state: LoanApplicationState) -> dict:
    """
    Specialist Agent 3: HR Risk Evaluator
    Evaluates employment stability and income reliability.
    Returns score 0-10 + reasoning + stability rating.
    """
    prompt = f"""
You are an HR risk evaluator at a bank.

Years at current employer : {state['employment_years']}
Employment type           : {state['employment_type']}

Risk guidance:
  Full-time / Government → Lower risk (stable income)
  Contract / Part-time   → Medium risk
  Self-employed          → Higher risk (variable income)

Evaluate the job stability and reliability of income.
Give a score between 0 and 10.
Give a one-line reasoning.

Respond ONLY in raw JSON. No markdown, no extra text:
{{"score": <number>, "reasoning": "<text>", "stability": "<STABLE/MODERATE/UNSTABLE>"}}
"""
    response = model.invoke(prompt)
    print(f"[Employment Agent] → {response.content}")
    return {"employment_result": response.content}

### Node 4 — Aggregator (Pure Logic — NO LLM)

This is the critical fix. The aggregator:
- Waits until ALL three parallel nodes have written their results into state
- Applies deterministic scoring rules — no LLM needed
- Produces a preliminary decision before passing to summarizer

**Why no LLM here?**
Aggregation is pure math + rules. Using an LLM here wastes tokens and introduces
non-determinism in a step that should be 100% predictable.

In [6]:
def aggregator(state: LoanApplicationState) -> dict:
    """
    Aggregator: Pure Logic — NO LLM call.
    Combines all three specialist results.
    Applies scoring rules to produce preliminary decision.

    Rules:
      - Any score < 4  → REJECT immediately
      - All scores ≥ 7 → APPROVE
      - Otherwise      → MANUAL_REVIEW
    """
    # Parse all three results
    try:
        credit = json.loads(state["credit_score_result"])
    except Exception:
        credit = {"score": 0, "reasoning": "Parse error", "risk_level": "HIGH"}

    try:
        income = json.loads(state["income_score_result"])
    except Exception:
        income = {"score": 0, "reasoning": "Parse error", "dti_ratio": "N/A"}

    try:
        employment = json.loads(state["employment_result"])
    except Exception:
        employment = {"score": 0, "reasoning": "Parse error", "stability": "UNSTABLE"}

    scores = [
        credit.get("score", 0),
        income.get("score", 0),
        employment.get("score", 0)
    ]

    # Apply deterministic rules
    if any(s < 4 for s in scores):
        decision = "REJECT"
    elif all(s >= 7 for s in scores):
        decision = "APPROVE"
    else:
        decision = "MANUAL_REVIEW"

    print(f"[Aggregator] Scores: {scores} → Preliminary Decision: {decision}")

    return {
        "credit_parsed": credit,
        "income_parsed": income,
        "employment_parsed": employment,
        "all_scores": scores,
        "preliminary_decision": decision
    }

### Node 5 — Summarizer (LLM)

Called ONCE after aggregation is complete.
Generates a human-readable verdict for the applicant.
This is the only place where natural language generation happens for the final output.

In [7]:
def summarizer(state: LoanApplicationState) -> dict:
    """
    Summarizer: LLM called ONCE.
    Takes aggregated scores and preliminary decision.
    Produces final verdict + human-readable explanation.
    """
    prompt = f"""
You are a senior loan approval officer writing to an applicant.

Applicant name     : {state['applicant_name']}
Preliminary decision: {state['preliminary_decision']}

Evaluation breakdown:
  Credit Assessment    : score {state['credit_parsed'].get('score')}/10
                         {state['credit_parsed'].get('reasoning')}
                         Risk level: {state['credit_parsed'].get('risk_level')}

  Income Assessment    : score {state['income_parsed'].get('score')}/10
                         {state['income_parsed'].get('reasoning')}
                         DTI ratio: {state['income_parsed'].get('dti_ratio')}

  Employment Assessment: score {state['employment_parsed'].get('score')}/10
                         {state['employment_parsed'].get('reasoning')}
                         Stability: {state['employment_parsed'].get('stability')}

Write a clear, professional, empathetic explanation of the final decision
directly addressing the applicant by name.
Confirm the preliminary decision — do not change it.
Highlight the key concern if decision is REJECT or MANUAL_REVIEW.

Respond ONLY in raw JSON. No markdown, no extra text:
{{"verdict": "<APPROVE/REJECT/MANUAL_REVIEW>",
  "summary": "<2-3 sentence explanation for applicant>",
  "key_concern": "<main risk factor or None if approved>"}}
"""
    response = model.invoke(prompt)
    result = json.loads(response.content)
    print(f"[Summarizer] → {result}")

    return {
        "verdict": result["verdict"],
        "summary": result["summary"],
        "key_concern": result["key_concern"]
    }

## Build and Compile the Graph

In [8]:
graph = StateGraph(LoanApplicationState)

# Register all nodes
graph.add_node("credit_evaluation",    credit_evaluation)
graph.add_node("income_verification",  income_verification)
graph.add_node("employment_evaluation", employment_evaluation)
graph.add_node("aggregator",           aggregator)
graph.add_node("summarizer",           summarizer)

# Fan-out: START triggers all three specialist agents in PARALLEL
graph.add_edge(START, "credit_evaluation")
graph.add_edge(START, "income_verification")
graph.add_edge(START, "employment_evaluation")

# Fan-in: all three feed into aggregator
# LangGraph waits for ALL incoming edges before triggering aggregator
graph.add_edge("credit_evaluation",    "aggregator")
graph.add_edge("income_verification",  "aggregator")
graph.add_edge("employment_evaluation", "aggregator")

# Sequential: aggregator → summarizer → END
graph.add_edge("aggregator",  "summarizer")
graph.add_edge("summarizer",  END)

workflow = graph.compile()
print("Graph compiled successfully.")

Graph compiled successfully.


## Run the Workflow

Test Case 1 — Poor credit applicant (expect REJECT)

In [9]:
application_poor = {
    "applicant_name":   "John Doe",
    "credit_score":     420,          # Poor — expect REJECT
    "loan_amount":      500000,
    "monthly_income":   40000,
    "employment_years": 2,
    "employment_type":  "part-time",
    # Initialise all output fields to empty — required by TypedDict
    "credit_score_result":  "",
    "income_score_result":  "",
    "employment_result":    "",
    "credit_parsed":        {},
    "income_parsed":        {},
    "employment_parsed":    {},
    "all_scores":           [],
    "preliminary_decision": "",
    "verdict":              "",
    "summary":              "",
    "key_concern":          ""
}

print("=" * 60)
print(f"Processing loan application for: {application_poor['applicant_name']}")
print("=" * 60)

result = workflow.invoke(application_poor)

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)
print(f"Verdict     : {result['verdict']}")
print(f"Summary     : {result['summary']}")
print(f"Key Concern : {result['key_concern']}")
print(f"All Scores  : Credit={result['all_scores'][0]}, Income={result['all_scores'][1]}, Employment={result['all_scores'][2]}")

Processing loan application for: John Doe
[Employment Agent] → {"score": 5, "reasoning": "Part-time employment is considered medium risk due to the variable income and potentially less stable job security.", "stability": "MODERATE"}
[Credit Agent] → {"score": 1, "reasoning": "Poor credit score indicates high risk of default", "risk_level": "HIGH"}
[Income Agent] → {"score": 9, "reasoning": "The applicant's DTI ratio is 20.83%, well below the industry safe threshold, indicating that the income adequately supports the loan.", "dti_ratio": "20.83%"}
[Aggregator] Scores: [1, 9, 5] → Preliminary Decision: REJECT
[Summarizer] → {'verdict': 'REJECT', 'summary': 'Unfortunately, after reviewing your application, we have decided to reject your loan request. The main concern is your poor credit score, which indicates a high risk of default.', 'key_concern': 'High risk due to poor credit score'}

FINAL RESULT
Verdict     : REJECT
Summary     : Unfortunately, after reviewing your application, we ha

Test Case 2 — Strong applicant (expect APPROVE)

In [10]:
application_good = {
    "applicant_name":   "Priya Sharma",
    "credit_score":     780,          # Excellent — expect APPROVE
    "loan_amount":      300000,
    "monthly_income":   120000,
    "employment_years": 8,
    "employment_type":  "full-time",
    "credit_score_result":  "",
    "income_score_result":  "",
    "employment_result":    "",
    "credit_parsed":        {},
    "income_parsed":        {},
    "employment_parsed":    {},
    "all_scores":           [],
    "preliminary_decision": "",
    "verdict":              "",
    "summary":              "",
    "key_concern":          ""
}

print("=" * 60)
print(f"Processing loan application for: {application_good['applicant_name']}")
print("=" * 60)

result2 = workflow.invoke(application_good)

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)
print(f"Verdict     : {result2['verdict']}")
print(f"Summary     : {result2['summary']}")
print(f"Key Concern : {result2['key_concern']}")
print(f"All Scores  : Credit={result2['all_scores'][0]}, Income={result2['all_scores'][1]}, Employment={result2['all_scores'][2]}")

Processing loan application for: Priya Sharma
[Credit Agent] → {"score": 9, "reasoning": "Excellent credit score and loan amount is well within the applicant's means", "risk_level": "LOW"}
[Income Agent] → {"score": 10, "reasoning": "The applicant's DTI ratio is 4.2%, well below the industry safe threshold of 40%, indicating that the income adequately supports the loan.", "dti_ratio": "4.2%"}
[Employment Agent] → {"score": 9, "reasoning": "Employee has been at current employer for 8 years, indicating job stability and reliable income.", "stability": "STABLE"}
[Aggregator] Scores: [9, 10, 9] → Preliminary Decision: APPROVE
[Summarizer] → {'verdict': 'APPROVE', 'summary': 'Dear Priya Sharma, I am pleased to inform you that your loan application has been approved. Your excellent credit score, low DTI ratio, and stable employment history were all key factors in our decision. Congratulations!', 'key_concern': 'None'}

FINAL RESULT
Verdict     : APPROVE
Summary     : Dear Priya Sharma, I am 